<a href="https://colab.research.google.com/github/virusds7778-coder/HOMEWORK_3/blob/main/HSE_Denis_Ivanov__final_hw_efrsb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Итоговое домашнее задание
# «Аналитик реестра — Обработка данных ЕФРСБ»





Вы — юрист, анализирующий данные из **Единого федерального реестра сведений о банкротстве (ЕФРСБ)**.
Вам поступила выгрузка в формате JSON с информацией о сообщениях по делам о банкротстве за 2025 год.

**Ваша задача** — извлечь ключевые данные, очистить от мусора, сопоставить с реестром организаций
и сформировать аналитический отчёт для руководства.

---
### 📋 Как выполнять и сдавать задание

**Где выполнять:**
- **Вариант 1 (рекомендуется):** Загрузите файл `.ipynb` в [Google Colab](https://colab.research.google.com/), загрузите папку `final_hw_data/` в среду выполнения (например, через боковую панель «Файлы» или смонтировав Google Drive).
- **Вариант 2:** Работайте локально в Jupyter Notebook / JupyterLab. Убедитесь, что папка `final_hw_data/` находится в той же директории, что и файл задания.

**Как сдавать на проверку:**
- Отправьте **ссылку на Google Colab** с общим доступом («Любой, у кого есть ссылка — комментатор»), **ИЛИ**
- Отправьте **ссылку на GitHub-репозиторий**, в котором лежит выполненный `.ipynb`-файл и папка `final_hw_data/` с результатами.

> ⚠️ Убедитесь, что ссылка открывается в режиме просмотра и все ячейки выполнены (результаты видны).

---

### 📦 Исходные данные

В папке `final_hw_data/` находятся 3 файла:

| Файл | Описание |
|------|----------|
| `bankruptcy_messages.json` | Сообщения ЕФРСБ |
| `organizations.json` | Реестр организаций |
| `priority_cases.txt` | Номера приоритетных дел |



---
# 🟢 Загрузка и кросс-референс (0-2 балла)
---

### Задание 1.1 — Загрузка данных из файлов

Загрузите данные из трёх файлов:
- `bankruptcy_messages.json` → список `messages`
- `organizations.json` → список `organizations`
- `priority_cases.txt` → список `priority_case_numbers`

Выведите количество загруженных записей из каждого источника.

In [17]:
import json

# Load bankruptcy_messages.json
with open('/content/bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

# Load organizations.json
with open('/content/organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

# Load priority_cases.txt
with open('/content/priority_cases.txt', 'r', encoding='utf-8') as f:
    priority_case_numbers = [line.strip() for line in f if line.strip()]

print(f"Загружено сообщений: {len(messages)}")
print(f"Загружено организаций: {len(organizations)}")
print(f"Загружено приоритетных номеров дел: {len(priority_case_numbers)}")

Загружено сообщений: 54
Загружено организаций: 30
Загружено приоритетных номеров дел: 10


### Задание 1.2 — Словарь организаций

Создайте словарь `org_by_inn`, где ключ — ИНН (строка), а значение — словарь с данными об организации.
Используйте **dict comprehension**.

Выведите количество организаций в словаре и информацию по ИНН `"7701234567"`.

In [18]:
org_by_inn = {org['inn']: org for org in organizations}

print(f"Количество организаций в словаре: {len(org_by_inn)}")
print(f"Информация по ИНН \"7701234567\": {org_by_inn.get('7701234567')}")

Количество организаций в словаре: 30
Информация по ИНН "7701234567": {'inn': '7701234567', 'ogrn': '1027700123456', 'name': 'ООО "Альфа-Строй"', 'address': 'г. Москва, ул. Строителей, д. 15', 'region': 'Москва'}


### Задание 1.3 — Кросс-референс: связываем сообщения с организациями

Для каждого сообщения добавьте поле `org_name` (название организации) и `region`,
используя словарь `org_by_inn` и метод `.get()`.
Если организация не найдена — ставьте значение `None`.

Посчитайте, сколько сообщений удалось связать с организацией,
а сколько — не удалось.

In [19]:
linked_messages_count = 0
unlinked_messages_count = 0

for message in messages:
    inn = message.get('publisher_inn')
    organization_info = org_by_inn.get(inn)

    if organization_info:
        message['org_name'] = organization_info.get('name')
        message['region'] = organization_info.get('region')
        linked_messages_count += 1
    else:
        message['org_name'] = None
        message['region'] = None
        unlinked_messages_count += 1

print(f"Сообщений, связанных с организацией: {linked_messages_count}")
print(f"Сообщений, не связанных с организацией: {unlinked_messages_count}")

Сообщений, связанных с организацией: 51
Сообщений, не связанных с организацией: 3


### Задание 1.4 — Множества и приоритетные дела

1. Создайте множество `priority_set` из списка `priority_case_numbers`.
2. Создайте множество `message_case_set` из номеров дел в `messages`
   (используйте list comprehension + filter для непустых номеров).
3. Найдите пересечение — номера дел, которые есть и в приоритетных, и в сообщениях (`&`).
4. Выведите результат.

In [20]:
priority_set = set(priority_case_numbers)

message_case_set = {msg['case_number'] for msg in messages if msg.get('case_number')}

intersection_cases = priority_set & message_case_set

print(f"Приоритетные дела: {priority_set}")
print(f"Номера дел из сообщений: {message_case_set}")
print(f"Пересечение приоритетных дел и дел из сообщений: {intersection_cases}")

Приоритетные дела: {'А60-22222/2025', 'А60-77777/2025', 'А60-56789/2025', 'А60-66666/2025', 'А60-44444/2025', 'А60-99999/2025', 'А60-11111/2025', 'А60-88888/2025', 'А60-33333/2025', 'А60-12345/2025'}
Номера дел из сообщений: {'А60-55555/2025', 'А60-22222/2025', 'А60-66666/2025', 'А60-56789/2025', 'А60-77777/2025', 'А60-25814/2025', 'А60-44444/2025', 'А60-99999/2025', 'А60-24680/2025', 'А60-11111/2025', 'А60-14702/2025', 'А60-36925/2025', 'А60-88888/2025', 'А60-33333/2025', 'А60-13579/2025', 'А60-12345/2025'}
Пересечение приоритетных дел и дел из сообщений: {'А60-22222/2025', 'А60-66666/2025', 'А60-56789/2025', 'А60-44444/2025', 'А60-99999/2025', 'А60-12345/2025', 'А60-33333/2025', 'А60-88888/2025', 'А60-11111/2025', 'А60-77777/2025'}


---
# 🟡 Очистка и валидация (0-3 балла)
---

### Задание 2.1 — Функция парсинга дат

Напишите функцию `parse_date(date_str)`, которая принимает строку с датой
и возвращает объект `datetime` или `None`, если распарсить не удалось.

Форматы для обработки:
- `"DD.MM.YYYY"` — `strptime`
- `"YYYY-MM-DD"` — `fromisoformat`
- `"DD месяца YYYY г."` — замена русских месяцев + `strptime`
- `"DD/MM/YYYY HH:MM"` — `strptime`

Обрабатывайте `None` и пустые строки через `try/except`.

In [21]:
from datetime import datetime

def parse_date(date_str):
    if not date_str:
        return None

    # Dictionary for Russian month names
    month_map = {
        'января': '01', 'февраля': '02', 'марта': '03', 'апреля': '04',
        'мая': '05', 'июня': '06', 'июля': '07', 'августа': '08',
        'сентября': '09', 'октября': '10', 'ноября': '11', 'декабря': '12'
    }

    # Try parsing different formats
    try:
        return datetime.strptime(date_str, '%d.%m.%Y')
    except ValueError:
        pass

    try:
        return datetime.fromisoformat(date_str)
    except ValueError:
        pass

    try:
        # Format: "DD месяца YYYY г."
        # Replace month name with its number
        parts = date_str.split()
        if len(parts) == 3 and parts[1] in month_map:
            day = parts[0]
            month = month_map[parts[1]]
            year = parts[2].replace('г.', '')
            formatted_date_str = f"{day}.{month}.{year.strip()}"
            return datetime.strptime(formatted_date_str, '%d.%m.%Y')
    except ValueError:
        pass

    try:
        return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
    except ValueError:
        pass

    return None


### Задание 2.2 — Функция валидации сообщения

Напишите функцию `validate_message(msg)`, которая:

1. Проверяет наличие **обязательных полей**: `publisher_inn`, `msg_text`, `date_published`, `type`, `case_number`.
   Если поле отсутствует — ошибка типа `KeyError`.
2. Проверяет, что **ИНН** состоит из 10 или 12 цифр (метод `.isdigit()`).
3. Парсит дату через `parse_date()`. Если дата не парсится — ошибка.
4. Проверяет, что **тип сообщения** — непустая строка.

Функция возвращает кортеж `(valid_msg, errors)`:
- `valid_msg` — словарь с очищенными данными (или `None`)
- `errors` — список строк с описанием ошибок

In [47]:
def validate_message(msg):
    errors = []
    processed_msg = msg.copy()

    # 1. Проверка наличия обязательных полей
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']
    for field in required_fields:
        if field not in processed_msg or processed_msg[field] is None:
            errors.append(f"KeyError: Отсутствует обязательное поле '{field}'")
            return None, errors # Immediate return if a required field is missing

    # If we are here, all required fields exist and are not None.
    # We can safely access them using processed_msg[field] directly.

    # 2. Проверка ИНН
    inn = str(processed_msg['publisher_inn'])
    if not (inn.isdigit() and (len(inn) == 10 or len(inn) == 12)):
        errors.append(f"ValueError: Некорректный ИНН '{inn}' (должен состоять из 10 или 12 цифр)")
        return None, errors

    # 3. Парсинг даты
    date_str = processed_msg['date_published']
    parsed_date = parse_date(date_str)
    if parsed_date is None:
        errors.append(f"ValueError: Не удалось распарсить дату '{date_str}'")
        return None, errors
    processed_msg['date_published'] = parsed_date # Update with parsed datetime object

    # 4. Проверка типа сообщения
    msg_type = processed_msg['type']
    if not isinstance(msg_type, str) or not msg_type.strip():
        errors.append(f"ValueError: Тип сообщения '{msg_type}' не должен быть пустым")
        return None, errors

    return processed_msg, []

### Задание 2.3 — Массовая валидация

Примените `validate_message()` ко всем сообщениям.
Разделите результат на два списка: `valid_messages` и `error_records`.

Соберите **статистику ошибок** по типам (сколько `KeyError`, `ValueError` и т.д.).

In [46]:
valid_messages = []
error_records = []
error_stats = {}

for msg in messages:
    validated_msg, errors = validate_message(msg)
    if validated_msg:
        valid_messages.append(validated_msg)
    else:
        error_records.append({
            'original_message': msg,
            'errors': errors
        })
        for error_str in errors:
            if error_str.startswith('KeyError'):
                error_stats['KeyError'] = error_stats.get('KeyError', 0) + 1
            elif error_str.startswith('ValueError'):
                error_stats['ValueError'] = error_stats.get('ValueError', 0) + 1
            else:
                error_stats['OtherError'] = error_stats.get('OtherError', 0) + 1

print(f"Всего сообщений: {len(messages)}")
print(f"Валидных сообщений: {len(valid_messages)}")
print(f"Сообщений с ошибками: {len(error_records)}")
print("Статистика ошибок:")
for error_type, count in error_stats.items():
    print(f"- {error_type}: {count}")

Всего сообщений: 54
Валидных сообщений: 48
Сообщений с ошибками: 6
Статистика ошибок:
- ValueError: 4
- KeyError: 2


---
# 🔵 Извлечение данных и аналитика (0-2 балла)
---

### Задание 3.1 — Извлечение сумм из текста

Напишите функцию `extract_amounts(text)`, которая ищет упоминания денежных сумм в тексте сообщения.

Используйте строковые методы.

Ищите по ключевым словам: `"руб."`, `"тыс. руб."`, `"млн руб."`.

Функция возвращает список строк — найденных сумм.

In [45]:
import re

def extract_amounts(text):
    amounts = []
    if not text:
        return amounts

    # Normalize text to handle various spacing and remove extra chars like periods after 'руб'
    text = text.lower().replace('.', '').replace(',', '')

    # Pattern for "X руб." (e.g., "15000000 руб")
    # Also covers cases like "15 000 000 руб"
    matches_rub = re.findall(r'(\d[\d\s]*) руб', text)
    for match in matches_rub:
        # Remove spaces and append
        amounts.append(match.replace(' ', ''))

    # Pattern for "X тыс. руб." (e.g., "8500 тыс руб")
    matches_thous = re.findall(r'(\d[\d\s]*) тыс руб', text)
    for match in matches_thous:
        # Convert to full rubles and append
        num_str = match.replace(' ', '')
        amounts.append(str(int(num_str) * 1000))

    # Pattern for "X млн руб." (e.g., "5 млн руб")
    matches_mln = re.findall(r'(\d[\d\s]*) млн руб', text)
    for match in matches_mln:
        # Convert to full rubles and append
        num_str = match.replace(' ', '')
        amounts.append(str(int(num_str) * 1000000))

    return amounts

### Задание 3.2 — Поиск упоминаний арбитражных судов

Напишите функцию `find_court_mentions(text)`, которая проверяет,
содержит ли текст упоминание арбитражного суда.

Ищите подстроку `"АС "` (с пробелом) в тексте через оператор `in`.
Верните `True` / `False`.

In [33]:
def find_court_mentions(text):
    if not text:
        return False
    return "АС " in text

### Задание 3.3 — Извлечение ФИО арбитражных управляющих

Напишите функцию `extract_manager_name(text)`, которая ищет ФИО арбитражного управляющего.

Алгоритм:
1. Проверьте, содержит ли текст подстроку `"арбитражный управляющий"`.
2. Если да — найдите позицию этой подстроки, возьмите текст после неё.
3. С помощью `.split()` извлеките следующие 3 слова (ФИО).
4. Соедините через пробел и верните.
5. Если не найдено — верните `None`.

In [44]:
def extract_manager_name(text):
    if not text:
        return None

    keyword = "арбитражный управляющий"
    if keyword in text:
        # Find the position of the keyword
        keyword_start_index = text.find(keyword)
        # Get the text after the keyword
        text_after_keyword = text[keyword_start_index + len(keyword):].strip()

        # Split the text into words and take the next 3 words (assuming FIO)
        words = text_after_keyword.split()
        if len(words) >= 3:
            # Join the first three words to form the name
            manager_name = ' '.join(words[:3])
            # Clean up any trailing punctuation if present
            manager_name = manager_name.replace(',', '').replace('.', '').strip()
            return manager_name
    return None

### Задание 3.4 — Обогащение данных

Для каждого **валидного** сообщения добавьте поля:
- `amounts` — список извлечённых сумм (функция `extract_amounts`)
- `has_court_mention` — `True`/`False` (функция `find_court_mentions`)
- `manager_name` — ФИО или `None` (функция `extract_manager_name`)

In [43]:
for message in valid_messages:
    message['amounts'] = extract_amounts(message.get('msg_text', ''))
    message['has_court_mention'] = find_court_mentions(message.get('msg_text', ''))
    message['manager_name'] = extract_manager_name(message.get('msg_text', ''))

# Optional: print a sample to verify
# print(valid_messages[0])
# print(valid_messages[1])

### Задание 3.5 — Аналитика

Постройте следующие показатели по **валидным** сообщениям:

1. **Количество сообщений по типам** — словарь `{тип: количество}`
2. **Количество сообщений по регионам** — словарь `{регион: количество}`, пропуская `None`
3. **Топ-5 публикаторов** по количеству сообщений — `sorted(key=lambda...)`
4. **Таймлайн** — сообщения, отсортированные по дате публикации

In [42]:
# 1. Количество сообщений по типам
type_stats = {}
for msg in valid_messages:
    msg_type = msg.get('type')
    if msg_type:
        type_stats[msg_type] = type_stats.get(msg_type, 0) + 1

print("Количество сообщений по типам:")
for msg_type, count in type_stats.items():
    print(f"- {msg_type}: {count}")

# 2. Количество сообщений по регионам
region_stats = {}
for msg in valid_messages:
    region = msg.get('region')
    if region:
        region_stats[region] = region_stats.get(region, 0) + 1

print("\nКоличество сообщений по регионам:")
for region, count in region_stats.items():
    print(f"- {region}: {count}")

# 3. Топ-5 публикаторов
publisher_counts = {}
for msg in valid_messages:
    publisher_inn = msg.get('publisher_inn')
    if publisher_inn:
        publisher_counts[publisher_inn] = publisher_counts.get(publisher_inn, 0) + 1

top_5_publishers_inn = sorted(publisher_counts.items(), key=lambda item: item[1], reverse=True)[:5]

top_5_publishers = []
for inn, count in top_5_publishers_inn:
    org_name = org_by_inn.get(inn, {}).get('name', f'Неизвестная организация ({inn})')
    top_5_publishers.append({'inn': inn, 'name': org_name, 'count': count})

print("\nТоп-5 публикаторов:")
for publisher in top_5_publishers:
    print(f"- {publisher['name']} (ИНН: {publisher['inn']}): {publisher['count']} сообщений")

# 4. Таймлайн — сообщения, отсортированные по дате публикации
timeline = sorted(valid_messages, key=lambda msg: msg['date_published'])

print("\nТаймлайн (первые 5 записей):")
for i, msg in enumerate(timeline[:5]):
    print(f"- {msg['date_published'].strftime('%d.%m.%Y')}: {msg.get('org_name', 'N/A')} - {msg.get('type', 'N/A')}")


Количество сообщений по типам:
- Введение процедуры: 8
- Собрание кредиторов: 4
- Завершение процедуры: 6
- Продажа имущества: 6
- Требование кредитора: 4
- Оспаривание сделки: 3
- Подача заявления: 3
- Отстранение управляющего: 1
- Мировое соглашение: 3
- Субсидиарная ответственность: 2
- Передача дела: 3
- Признание банкротом: 2
- Назначение управляющего: 2
- Жалоба на управляющего: 1

Количество сообщений по регионам:
- Москва: 26
- Свердловская область: 6
- Краснодарский край: 3
- Челябинская область: 3
- Ростовская область: 2
- Санкт-Петербург: 4
- Владимирская область: 1
- Московская область: 2

Топ-5 публикаторов:
- ООО "Альфа-Строй" (ИНН: 7701234567): 3 сообщений
- ООО "Тета-Консалтинг" (ИНН: 7706567890): 3 сообщений
- ООО "Кси-Ритейл" (ИНН: 7703901234): 3 сообщений
- ЗАО "Бета-Инвест" (ИНН: 7702345678): 2 сообщений
- ООО "Гамма-Трейд" (ИНН: 6658123450): 2 сообщений

Таймлайн (первые 5 записей):
- 10.01.2025: ООО "Гамма-Трейд" - Завершение процедуры
- 15.01.2025: ООО "Альфа-Стр

---
# 🟣 Отчётность (0-1 балл)
---

### Задание 4.1 — Подготовка данных для сохранения

Подготовьте данные для записи в JSON. Даты нужно преобразовать обратно в строки (`strftime`),
чтобы JSON был читаемым.

In [41]:
prepared_valid_messages = []
for msg in valid_messages:
    # Create a copy to avoid modifying the original valid_messages list
    # if it's needed in its original datetime format later
    prepared_msg = msg.copy()
    if 'date_published' in prepared_msg and isinstance(prepared_msg['date_published'], datetime):
        prepared_msg['date_published'] = prepared_msg['date_published'].strftime('%Y-%m-%d %H:%M:%S')
    prepared_valid_messages.append(prepared_msg)

# Also prepare error_records for JSON output if they contain datetime objects
prepared_error_records = []
for err_rec in error_records:
    prepared_err_rec = err_rec.copy()
    original_msg = prepared_err_rec.get('original_message')
    if original_msg and 'date_published' in original_msg and isinstance(original_msg['date_published'], datetime):
        original_msg['date_published'] = original_msg['date_published'].strftime('%Y-%m-%d %H:%M:%S')
    prepared_error_records.append(prepared_err_rec)

print("Данные подготовлены для сохранения. Пример даты в первом валидном сообщении:")
if prepared_valid_messages:
    print(prepared_valid_messages[0].get('date_published'))


Данные подготовлены для сохранения. Пример даты в первом валидном сообщении:
2025-01-15 00:00:00


### Задание 4.2 — Сохранение `analysis_results.json`

Сохраните файл `analysis_results.json` со следующей структурой:
```json
{
  "valid_messages": [...],
  "type_stats": {...},
  "region_stats": {...},
  "priority_messages": [...]
}
```

In [40]:
analysis_results = {
    "valid_messages": prepared_valid_messages,
    "type_stats": type_stats,
    "region_stats": region_stats,
    "priority_messages": [msg for msg in prepared_valid_messages if msg.get('case_number') in priority_set]
}

with open('analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

print("Файл 'analysis_results.json' успешно сохранен.")

Файл 'analysis_results.json' успешно сохранен.


### Задание 4.3 — Сохранение `validation_errors.json`

Сохраните проблемные записи с описанием ошибок.

In [39]:
with open('validation_errors.json', 'w', encoding='utf-8') as f:
    json.dump(prepared_error_records, f, ensure_ascii=False, indent=2)

print("Файл 'validation_errors.json' успешно сохранен.")

Файл 'validation_errors.json' успешно сохранен.


### Задание 4.4 — Текстовый отчёт `summary_report.txt`

Сформируйте текстовый аналитический отчёт для руководства с помощью **f-строк**.

Отчёт должен содержать:
- Заголовок и дату
- Общую статистику (всего сообщений, валидных, ошибочных)
- Статистику по типам сообщений
- Статистику по регионам
- Топ-5 публикаторов
- Список приоритетных дел
- Список найденных арбитражных управляющих

In [38]:
from datetime import datetime

# Get current date for the report header
report_date = datetime.now().strftime('%d.%m.%Y %H:%M:%S')

# Collect arbitration managers who were actually found
found_managers = {msg['manager_name'] for msg in valid_messages if msg['manager_name'] is not None}

# Create the report content
report_content = f"""# Аналитический отчёт по сообщениям о банкротстве\nДата формирования отчёта: {report_date}\n\n--- Общая статистика ---\nВсего сообщений: {len(messages)}\nВалидных сообщений: {len(valid_messages)}\nСообщений с ошибками: {len(error_records)}\n\n--- Статистика ошибок ---\n"""
for error_type, count in error_stats.items():
    report_content += f"- {error_type}: {count}\n"

report_content += f"""\n--- Количество сообщений по типам ---\n"""
for msg_type, count in type_stats.items():
    report_content += f"- {msg_type}: {count}\n"

report_content += f"""\n--- Количество сообщений по регионам ---\n"""
for region, count in region_stats.items():
    report_content += f"- {region}: {count}\n"

report_content += f"""\n--- Топ-5 публикаторов ---\n"""
for publisher in top_5_publishers:
    report_content += f"- {publisher['name']} (ИНН: {publisher['inn']}): {publisher['count']} сообщений\n"

report_content += f"""\n--- Приоритетные дела (найденные в сообщениях) ---\n"""
if intersection_cases:
    for case in sorted(list(intersection_cases)):
        report_content += f"- {case}\n"
else:
    report_content += "- Приоритетных дел, найденных в сообщениях, нет.\n"

report_content += f"""\n--- Обнаруженные арбитражные управляющие ---\n"""
if found_managers:
    for manager in sorted(list(found_managers)):
        report_content += f"- {manager}\n"
else:
    report_content += "- Арбитражные управляющие не найдены.\n"


# Save the report to a text file
with open('summary_report.txt', 'w', encoding='utf-8') as f:
    f.write(report_content)

print("Файл 'summary_report.txt' успешно сохранен.")

Файл 'summary_report.txt' успешно сохранен.


---
## ✅ Итоги

Если вы корректно выполнили все 4 уровня, у вас должны получиться:

| Файл | Описание |
|------|----------|
| `analysis_results.json` | Валидные сообщения + статистика + приоритетные дела |
| `validation_errors.json` | Проблемные записи с описанием ошибок |
| `summary_report.txt` | Текстовый аналитический отчёт для руководства |

